In [7]:
import pandas as pd
import time
import os
from serpapi.google_search import GoogleSearch

SERPAPI_KEY = "f5a9defee71355d186e3110afdae81a2f604a8688fae9fc4dd25e12da0e12c8d"

KEYWORDS  = ["bitcoin", "USDT", "crypto"]
COUNTRIES = {
    "Kenya":        "KE",
    "Nigeria":      "NG",
    "Ghana":        "GH",
    "South Africa": "ZA"
}
TIMEFRAME  = "today 52-w"
OUTPUT_DIR = "../data"

print("Configuration ready.")
print(f"Keywords:  {KEYWORDS}")
print(f"Countries: {list(COUNTRIES.keys())}")

Configuration ready.
Keywords:  ['bitcoin', 'USDT', 'crypto']
Countries: ['Kenya', 'Nigeria', 'Ghana', 'South Africa']


In [13]:
import datetime

def pull_trends_serpapi(keyword, geo_code, country_name):
    try:
        # SerpApi requires YYYY-MM-DD format
        end_date   = datetime.date.today()
        start_date = end_date - datetime.timedelta(weeks=52)
        date_range = f"{start_date.strftime('%Y-%m-%d')} {end_date.strftime('%Y-%m-%d')}"

        params = {
            "engine":    "google_trends",
            "q":         keyword,
            "geo":       geo_code,
            "date":      date_range,
            "data_type": "TIMESERIES",
            "api_key":   SERPAPI_KEY
        }

        search   = GoogleSearch(params)
        results  = search.get_dict()
        timeline = results.get("interest_over_time", {}).get("timeline_data", [])

        if not timeline:
            print(f"  No data: {keyword} / {country_name}")
            return None

        rows = []
        for point in timeline:
            # Parse "Jun 22 – 28, 2025" — take the start date
            raw_date = point["date"].replace("\u2009", " ").replace("\u200b", "")
            start    = raw_date.split("–")[0].strip()

            # Handle "Jun 22, 2025" and "Jun 22 2025"
            try:
                week = pd.to_datetime(start + f" {end_date.year}"
                       if len(start.split()) == 2 else start)
            except Exception:
                week = pd.to_datetime(point["timestamp"], unit="s")

            rows.append({
                "week":     week,
                "interest": int(point["values"][0]["extracted_value"]),
                "keyword":  keyword,
                "country":  country_name,
                "geo_code": geo_code
            })

        df = pd.DataFrame(rows)
        df["week"] = pd.to_datetime(df["week"])
        print(f"  ✓ {len(df)} weeks — {keyword} / {country_name}")
        return df

    except Exception as e:
        print(f"  Failed: {keyword} / {country_name} — {e}")
        return None

print("Function defined.")

Function defined.


In [14]:
print("Testing: bitcoin / Kenya...")
df_test = pull_trends_serpapi("bitcoin", "KE", "Kenya")

if df_test is not None:
    print(f"\nShape: {df_test.shape}")
    print(df_test.head(3))
else:
    print("Failed.")

Testing: bitcoin / Kenya...
  ✓ 53 weeks — bitcoin / Kenya

Shape: (53, 5)
        week  interest  keyword country geo_code
0 2026-06-22        42  bitcoin   Kenya       KE
1 2026-06-29        44  bitcoin   Kenya       KE
2 2026-07-06        43  bitcoin   Kenya       KE


In [15]:
all_results = []
total       = len(COUNTRIES) * len(KEYWORDS)
count       = 0

for country_name, geo_code in COUNTRIES.items():
    for keyword in KEYWORDS:
        count += 1
        print(f"[{count}/{total}] {keyword} / {country_name}...")

        df = pull_trends_serpapi(keyword, geo_code, country_name)

        if df is not None:
            all_results.append(df)

        if count < total:
            time.sleep(15)

print(f"\nDone. {len(all_results)} of {total} successful.")

[1/12] bitcoin / Kenya...
  ✓ 53 weeks — bitcoin / Kenya
[2/12] USDT / Kenya...
  ✓ 53 weeks — USDT / Kenya
[3/12] crypto / Kenya...
  ✓ 53 weeks — crypto / Kenya
[4/12] bitcoin / Nigeria...
  ✓ 53 weeks — bitcoin / Nigeria
[5/12] USDT / Nigeria...
  ✓ 53 weeks — USDT / Nigeria
[6/12] crypto / Nigeria...
  ✓ 53 weeks — crypto / Nigeria
[7/12] bitcoin / Ghana...
  ✓ 53 weeks — bitcoin / Ghana
[8/12] USDT / Ghana...
  ✓ 53 weeks — USDT / Ghana
[9/12] crypto / Ghana...
  ✓ 53 weeks — crypto / Ghana
[10/12] bitcoin / South Africa...
  ✓ 53 weeks — bitcoin / South Africa
[11/12] USDT / South Africa...
  ✓ 53 weeks — USDT / South Africa
[12/12] crypto / South Africa...
  ✓ 53 weeks — crypto / South Africa

Done. 12 of 12 successful.
